# Zajęcia 5 — śledzenie eksperymentów i wybór modelu

Notebook wykonuje serię kontrolowanych eksperymentów z pełnym zapisem kontekstu i kończy się udokumentowanym wyborem modelu.

Serwer śledzenia działa lokalnie, na bazie SQLite w katalogu artefaktów. Nie wymaga konta ani usługi zewnętrznej. Interfejs graficzny uruchamia się poleceniem:

```bash
mlflow ui --backend-store-uri sqlite:///artifacts/zajecia_05/mlflow.db
```

Komórki opisane jako **zadanie** wypełniasz samodzielnie. Komórka pod nimi zawiera rozwiązanie ukryte magią `%%skip True`.

## 1. Przygotowanie notebooka

Ustawiamy lokalny backend śledzenia i jawną lokalizację artefaktów, żeby wszystko powstawało w `artifacts/zajecia_05`. Ta komórka jest gotowa — uruchom ją bez zmian.

In [ ]:
import hashlib
import json
import os
import platform
import shutil
import subprocess
import sys
import time
from pathlib import Path

os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"

import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from mlflow.models import infer_signature
from sklearn.datasets import load_iris
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "skip_kernel_extension.py").exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

BASE_DIR = REPO_ROOT / "artifacts" / "zajecia_05"
REPORTS_DIR = BASE_DIR / "reports"
ARTIFACT_DIR = BASE_DIR / "mlartifacts"
for directory in (REPORTS_DIR, ARTIFACT_DIR):
    directory.mkdir(parents=True, exist_ok=True)

TRACKING_URI = "sqlite:///" + (BASE_DIR / "mlflow.db").as_posix()
EXPERIMENT_NAME = "zajecia_05_iris"

# Notebook zaczyna od czystej historii, zeby wynik nie zalezal od poprzednich uruchomien.
if (BASE_DIR / "mlflow.db").exists():
    (BASE_DIR / "mlflow.db").unlink()
    shutil.rmtree(ARTIFACT_DIR, ignore_errors=True)
    ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

mlflow.set_tracking_uri(TRACKING_URI)
if mlflow.get_experiment_by_name(EXPERIMENT_NAME) is None:
    mlflow.create_experiment(EXPERIMENT_NAME, artifact_location=ARTIFACT_DIR.as_uri())
mlflow.set_experiment(EXPERIMENT_NAME)

print(f"Backend śledzenia: {TRACKING_URI}")
print(f"Artefakty: {ARTIFACT_DIR.relative_to(REPO_ROOT)}")

%load_ext skip_kernel_extension

## 2. Trzy zbiory, trzy role

Dzielimy dane na trening, walidację i test. Zbiór testowy odkładamy i **nie dotykamy go** do momentu, w którym model będzie już wybrany. Ta komórka jest gotowa — uruchom ją bez zmian.

In [ ]:
SEED = 42
FEATURE_COLUMNS = [
    "sepal length (cm)",
    "sepal width (cm)",
    "petal length (cm)",
    "petal width (cm)",
]
TARGET_COLUMN = "target"

DATA_PATH = BASE_DIR / "iris.csv"
load_iris(as_frame=True).frame.to_csv(DATA_PATH, index=False)
data = pd.read_csv(DATA_PATH)

features = data[FEATURE_COLUMNS]
target = data[TARGET_COLUMN]

x_pool, x_test, y_pool, y_test = train_test_split(
    features, target, test_size=0.2, random_state=SEED, stratify=target
)
x_train, x_valid, y_train, y_valid = train_test_split(
    x_pool, y_pool, test_size=0.25, random_state=SEED, stratify=y_pool
)

print(f"trening:   {len(x_train)}")
print(f"walidacja: {len(x_valid)}")
print(f"test:      {len(x_test)}  <- używamy dokładnie raz, na końcu")

## 3. Kontekst uruchomienia

Metryka bez kontekstu jest bezużyteczna. Do każdego przebiegu dołączamy wersję danych (suma kontrolna z zajęć 3), wersję kodu i wersje bibliotek.

**Zadanie:** napisz funkcję `run_context()` zwracającą słownik tagów: `data_sha256`, `data_rows`, `git_commit` (albo `"brak"`, gdy repozytorium nie jest dostępne), `python_version` i `sklearn_version`.

In [ ]:
# Zadanie: zbuduj słownik tagów opisujących kontekst uruchomienia.
# Wskazówka: identyfikator commita uzyskasz z `git rev-parse --short HEAD`,
# ale komórka nie może się wywrócić, gdy git jest niedostępny.


In [ ]:
%%skip True
def file_sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(65536), b""):
            digest.update(chunk)
    return digest.hexdigest()


def git_commit() -> str:
    if shutil.which("git") is None:
        return "brak"
    result = subprocess.run(
        ["git", "rev-parse", "--short", "HEAD"],
        cwd=REPO_ROOT,
        capture_output=True,
        text=True,
    )
    return result.stdout.strip() if result.returncode == 0 else "brak"


def run_context() -> dict:
    return {
        "data_sha256": file_sha256(DATA_PATH)[:16],
        "data_rows": str(len(data)),
        "git_commit": git_commit(),
        "python_version": platform.python_version(),
        "sklearn_version": sklearn.__version__,
    }


CONTEXT = run_context()
print(json.dumps(CONTEXT, indent=2))

## 4. Pojedynczy przebieg

Przebieg zapisuje parametry, metryki, tagi i artefakty. Model logujemy razem z **sygnaturą** — opisem oczekiwanego wejścia i wyjścia. Do sygnatur wrócimy na zajęciach 7, tutaj zaczynamy je zapisywać.

Adres zalogowanego modelu zapisujemy dodatkowo jako **tag**. Dzięki temu tabela porównawcza zbudowana z backendu śledzenia będzie zawierać wszystko, czego potrzeba do wczytania modelu.

**Zadanie:** napisz funkcję `log_experiment(name, model, params)`, która trenuje model na zbiorze treningowym, liczy metryki na walidacyjnym i zapisuje cały przebieg. Zwróć słownik z nazwą, metrykami i identyfikatorem przebiegu.

In [ ]:
# Zadanie: zaloguj pojedynczy eksperyment.
# Wskazówka: mlflow.start_run, log_params, log_metrics, set_tags,
# mlflow.sklearn.log_model(sk_model=..., name="model", signature=..., input_example=...).


In [ ]:
%%skip True
def log_experiment(name: str, model, params: dict) -> dict:
    with mlflow.start_run(run_name=name) as run:
        started = time.perf_counter()
        model.fit(x_train, y_train)
        fit_seconds = time.perf_counter() - started

        predictions = model.predict(x_valid)
        metrics = {
            "valid_accuracy": float(accuracy_score(y_valid, predictions)),
            "valid_f1_macro": float(f1_score(y_valid, predictions, average="macro")),
            "fit_seconds": float(fit_seconds),
        }

        mlflow.log_params({"model": name, **params, "seed": SEED})
        mlflow.log_metrics(metrics)
        mlflow.set_tags(CONTEXT)

        signature = infer_signature(x_train, model.predict(x_train))
        info = mlflow.sklearn.log_model(
            sk_model=model,
            name="model",
            signature=signature,
            input_example=x_train.head(3),
        )
        mlflow.set_tag("model_uri", info.model_uri)

        metrics_path = REPORTS_DIR / f"metrics_{name}.json"
        metrics_path.write_text(json.dumps(metrics, indent=2), encoding="utf-8")
        mlflow.log_artifact(str(metrics_path))

        return {"name": name, "run_id": run.info.run_id, "model_uri": info.model_uri, **metrics}


result = log_experiment("baseline", DummyClassifier(strategy="most_frequent"), {})
print(json.dumps(result, indent=2))

## 5. Seria kontrolowanych eksperymentów

Eksperyment jest kontrolowany, gdy między przebiegami zmienia się dokładnie jedna rzecz. Wszystkie warianty korzystają z tego samego podziału danych i tego samego ziarna.

**Zadanie:** uruchom wszystkie konfiguracje z listy `CONFIGS` i zbierz wyniki do listy.

In [ ]:
def logistic(regularization: float) -> Pipeline:
    return Pipeline(
        steps=[
            ("scaler", StandardScaler()),
            ("classifier", LogisticRegression(C=regularization, max_iter=500, random_state=SEED)),
        ]
    )


CONFIGS = [
    ("logreg_C1", logistic(1.0), {"C": 1.0}),
    ("logreg_C005", logistic(0.05), {"C": 0.05}),
    (
        "random_forest",
        RandomForestClassifier(n_estimators=200, max_depth=3, random_state=SEED),
        {"n_estimators": 200, "max_depth": 3},
    ),
]

# Zadanie: uruchom wszystkie konfiguracje i zbierz wyniki.


In [ ]:
%%skip True
results = [result]
for name, model, params in CONFIGS:
    results.append(log_experiment(name, model, params))

display(pd.DataFrame(results))

## 6. Tabela porównawcza

Tabelę budujemy z zapisanych przebiegów, a nie z pamięci notebooka. To ważne: po ponownym uruchomieniu jądra dane nadal są dostępne, bo mieszkają w backendzie śledzenia.

**Zadanie:** pobierz przebiegi funkcją `mlflow.search_runs`, zostaw kolumny z nazwą modelu, metrykami, wersją danych i adresem modelu, posortuj po macro F1 i zapisz wynik do `reports/comparison.csv`.

In [ ]:
# Zadanie: zbuduj tabelę porównawczą z zapisanych przebiegów.


In [ ]:
%%skip True
runs = mlflow.search_runs(experiment_names=[EXPERIMENT_NAME])
comparison = (
    runs[
        [
            "params.model",
            "metrics.valid_f1_macro",
            "metrics.valid_accuracy",
            "metrics.fit_seconds",
            "tags.data_sha256",
            "tags.model_uri",
            "run_id",
        ]
    ]
    .rename(
        columns={
            "params.model": "model",
            "metrics.valid_f1_macro": "valid_f1_macro",
            "metrics.valid_accuracy": "valid_accuracy",
            "metrics.fit_seconds": "fit_seconds",
            "tags.data_sha256": "data_version",
            "tags.model_uri": "model_uri",
        }
    )
    .dropna(subset=["model"])
    .sort_values("valid_f1_macro", ascending=False)
    .reset_index(drop=True)
)

comparison.to_csv(REPORTS_DIR / "comparison.csv", index=False)
display(comparison)

## 7. Kryterium wyboru

Kryterium zapisujemy **przed** spojrzeniem na wyniki:

> Wybieram model o najwyższym macro F1 na zbiorze walidacyjnym. Przy różnicy mniejszej niż 0,01 wybieram model o krótszym czasie treningu. Model musi przewyższyć baseline o co najmniej 0,05 macro F1.

**Zadanie:** zaimplementuj to kryterium jako funkcję `select_model(comparison)` zwracającą wiersz zwycięzcy i zgłaszającą wyjątek, gdy żaden model nie spełnia warunku brzegowego.

In [ ]:
# Zadanie: zaimplementuj kryterium wyboru modelu.


In [ ]:
%%skip True
MIN_ADVANTAGE = 0.05
TIE_TOLERANCE = 0.01


def select_model(table: pd.DataFrame) -> pd.Series:
    baseline_score = table.loc[table["model"] == "baseline", "valid_f1_macro"].max()
    candidates = table[table["model"] != "baseline"].copy()

    eligible = candidates[candidates["valid_f1_macro"] >= baseline_score + MIN_ADVANTAGE]
    if eligible.empty:
        raise ValueError(
            f"Żaden model nie przewyższa baseline ({baseline_score:.3f}) o {MIN_ADVANTAGE}."
        )

    best_score = eligible["valid_f1_macro"].max()
    tied = eligible[eligible["valid_f1_macro"] >= best_score - TIE_TOLERANCE]
    return tied.sort_values("fit_seconds").iloc[0]


winner = select_model(comparison)
print(f"Wybrany model: {winner['model']} (valid F1 = {winner['valid_f1_macro']:.4f})")
print(f"Przebieg: {winner['run_id']}")

## 8. Jedno spojrzenie na zbiór testowy

Model jest już wybrany, więc możemy oszacować jego jakość na zbiorze testowym. To jedyne użycie tego zbioru w całym notebooku.

**Zadanie:** wczytaj model zwycięzcy z backendu śledzenia, policz metryki na zbiorze testowym i zapisz notatkę `reports/selection.md` z uzasadnieniem wyboru.

In [ ]:
# Zadanie: oceń wybrany model na zbiorze testowym i zapisz uzasadnienie.
# Wskazówka: adres modelu masz w kolumnie `model_uri`; użyj mlflow.sklearn.load_model.


In [ ]:
%%skip True
best_model = mlflow.sklearn.load_model(winner["model_uri"])
test_predictions = best_model.predict(x_test)
test_metrics = {
    "test_accuracy": round(float(accuracy_score(y_test, test_predictions)), 4),
    "test_f1_macro": round(float(f1_score(y_test, test_predictions, average="macro")), 4),
}

with mlflow.start_run(run_id=winner["run_id"]):
    mlflow.log_metrics(test_metrics)
    mlflow.set_tag("selected", "true")

selection = f"""# Wybór modelu — zajęcia 5

## Kryterium (zapisane przed eksperymentami)

Najwyższe macro F1 na zbiorze walidacyjnym; przy różnicy poniżej {TIE_TOLERANCE}
wygrywa model o krótszym czasie treningu; wymagana przewaga nad baseline: {MIN_ADVANTAGE}.

## Wynik

- Wybrany model: **{winner['model']}**
- Macro F1 (walidacja): {winner['valid_f1_macro']:.4f}
- Macro F1 (test, jednorazowo): {test_metrics['test_f1_macro']:.4f}
- Accuracy (test): {test_metrics['test_accuracy']:.4f}
- Przebieg: `{winner['run_id']}`
- Adres modelu: `{winner['model_uri']}`
- Wersja danych: `{winner['data_version']}`
- Commit: `{CONTEXT['git_commit']}`

## Uzasadnienie

Model spełnia warunek przewagi nad rozwiązaniem bazowym i wygrywa według metryki głównej.
Wynik na zbiorze testowym policzono jeden raz, po podjęciu decyzji.
"""

(REPORTS_DIR / "selection.md").write_text(selection, encoding="utf-8")
print(selection)

## 9. Punkt kontrolny 1 — co się dzieje, gdy wybierasz na teście

Porównujemy dwie strategie: wybór według walidacji (poprawnie) i wybór według testu (błąd metodologiczny).

**Do opisania:** o ile deklarowana jakość różni się od rzeczywistej i dlaczego przy dwudziestu konfiguracjach ta różnica byłaby większa niż przy czterech?

In [ ]:
rows = []
for _, row in comparison.iterrows():
    model = mlflow.sklearn.load_model(row["model_uri"])
    rows.append(
        {
            "model": row["model"],
            "valid_f1_macro": round(row["valid_f1_macro"], 4),
            "test_f1_macro": round(
                float(f1_score(y_test, model.predict(x_test), average="macro")), 4
            ),
        }
    )

both = pd.DataFrame(rows)
display(both)

by_valid = both.sort_values("valid_f1_macro", ascending=False).iloc[0]
by_test = both.sort_values("test_f1_macro", ascending=False).iloc[0]

print(f"Wybór wg walidacji: {by_valid['model']}, deklarowany wynik {by_valid['valid_f1_macro']}")
print(f"Wybór wg testu:     {by_test['model']}, deklarowany wynik {by_test['test_f1_macro']}")
print("\nWybór wg testu zawsze wygląda lepiej — bo zawiera składnik szczęścia.")

## 10. Punkt kontrolny 2 — rozrzut wyniku

Uruchamiamy ten sam wariant modelu z różnymi ziarnami podziału danych.

**Do opisania:** jaki jest rozrzut metryki i czy różnica między dwoma modelami z tabeli mieści się w tym rozrzucie?

In [ ]:
scores = []
for seed in range(10):
    a_train, a_valid, b_train, b_valid = train_test_split(
        x_pool, y_pool, test_size=0.25, random_state=seed, stratify=y_pool
    )
    model = logistic(1.0).fit(a_train, b_train)
    scores.append(f1_score(b_valid, model.predict(a_valid), average="macro"))

scores = np.array(scores)
print(f"Średnia:        {scores.mean():.4f}")
print(f"Odchylenie:     {scores.std():.4f}")
print(f"Zakres:         {scores.min():.4f} – {scores.max():.4f}")
print(f"Rozpiętość:     {scores.max() - scores.min():.4f}")

## 11. Punkt kontrolny 3 — autologowanie

Automatyczne logowanie zapisuje parametry i metryki bez pisania kodu.

**Do opisania:** ile parametrów zapisało się automatycznie i czego wśród nich brakuje w porównaniu z przebiegami logowanymi ręcznie?

In [ ]:
mlflow.sklearn.autolog(log_models=False)

with mlflow.start_run(run_name="autolog_demo") as auto_run:
    logistic(1.0).fit(x_train, y_train)

mlflow.sklearn.autolog(disable=True)

logged = mlflow.get_run(auto_run.info.run_id)
print(f"Zapisane automatycznie parametry: {len(logged.data.params)}")
print(f"Przykłady: {list(logged.data.params)[:6]}")
print(f"\nTagi kontekstu (data_sha256): {logged.data.tags.get('data_sha256', 'BRAK')}")

## 12. Podsumowanie

Eksperymenty przestały być jednorazowe. Każdy przebieg zostawia trwały ślad: parametry, metryki, artefakty i kontekst. Wybór modelu wynika z zapisanego wcześniej kryterium, a zbiór testowy został użyty raz.

Czego nadal brakuje:

- nic nie sprawdza automatycznie, czy kod i model działają poprawnie (zajęcia 6);
- model jest zalogowany, ale nie zarejestrowany — nie ma wersji, aliasu ani karty modelu (zajęcia 7);
- nie da się go jeszcze wywołać spoza notebooka (zajęcia 8).

### Zadanie do własnego projektu

1. Śledzenie eksperymentów podłączone do potoku treningowego.
2. Tagi z wersją danych, commitem i wersjami bibliotek w każdym przebiegu.
3. Podział na trzy zbiory albo walidacja krzyżowa z osobnym zbiorem testowym.
4. Kryterium wyboru zapisane w dokumentacji przed eksperymentami.
5. Co najmniej trzy kontrolowane eksperymenty, w tym model bazowy.
6. Tabela porównawcza w repozytorium.
7. Uzasadnienie wyboru odwołujące się do kryterium.